
# Pipeline OCR/VLM - Domiciliation des salaires étrangers

**Modèle : Qwen2.5-VL-7B-Instruct sur GPU Domino**

Ce notebook est inspiré de `pipeline ocr v12.ipynb`, avec une architecture dédiée aux dossiers de domiciliation :

- classification GPU des pages ;
- extraction spécialisée par type de document ;
- extraction KYC étendue, notamment le père et la mère ;
- conservation des valeurs brutes et normalisées ;
- génération d'un JSON par dossier ;
- export Excel avec les onglets `DOMICILIATIONS`, `PLANNING_TL`, `DOCUMENTS`, `CHAMPS_SOURCE`, `ERREURS` et `PARAMETRES` ;
- gestion des renouvellements et des mois scindés en `P1` / `P2` ;
- checkpoint après chaque dossier pour reprise après incident.

Le pipeline **n'effectue pas les contrôles réglementaires finaux**. Les règles métier et les décisions restent dans Alteryx.


## 1. Dépendances

In [ ]:

import sys
from importlib import metadata

REQUIRED_PACKAGES = {
    "torch": "2.0",
    "transformers": "4.45",
    "accelerate": "0.30",
    "PyMuPDF": "1.23",
    "Pillow": "9.0",
    "openpyxl": "3.1",
    "pandas": "1.5",
    "psutil": "5.9",
}

print("Python :", sys.version.replace("\n", " "))
print("\nPackages détectés :")
missing = []
for package_name, minimum in REQUIRED_PACKAGES.items():
    try:
        version = metadata.version(package_name)
        print(f"  {package_name:15s} {version:12s} | minimum conseillé {minimum}")
    except metadata.PackageNotFoundError:
        missing.append(package_name)
        print(f"  {package_name:15s} ABSENT")

if missing:
    raise RuntimeError(
        "Packages manquants : " + ", ".join(missing) +
        ". Installer uniquement ces packages dans l'environnement Domino."
    )

print("\n✅ Vérification des packages terminée sans modification de l'environnement")


## 2. Imports

In [ ]:

import gc
import hashlib
import json
import math
import re
import sys
import time
import calendar
from collections import defaultdict
from datetime import date, datetime, timedelta
from pathlib import Path

import fitz
import numpy as np
import pandas as pd
import psutil
import torch
from PIL import Image
from openpyxl import Workbook
from openpyxl.styles import Alignment, Border, Font, PatternFill, Side
from openpyxl.utils import get_column_letter
from transformers import AutoProcessor, AutoModelForVision2Seq

print("✅ Imports OK")
print("Python       :", sys.version.split()[0])
print("PyMuPDF     :", fitz.__doc__.splitlines()[0] if fitz.__doc__ else "chargé")
print("Torch       :", torch.__version__)
print("CUDA dispo  :", torch.cuda.is_available())
print("GPU         :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Aucun")


## 3. Configuration

In [ ]:
MODEL_PATH = '/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen2.5-VL-7B-Instruct/main'
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
PDF_ZOOM = 3.0
IMAGE_MAX_SIZE = 2400
BLANK_THRESHOLD = 0.995
GPU_BATCH_SIZE_CLASSIFICATION = 3
GPU_BATCH_SIZE_EXTRACTION = 2
MAX_NEW_TOKENS_CLASSIFICATION = 100
MAX_NEW_TOKENS_EXTRACTION = 1700
INPUT_DIR = Path('/mnt/data/domiciliations_in')
OUTPUT_DIR = Path('/mnt/data/domiciliations_out')
JSON_DIR = OUTPUT_DIR / 'json_dossiers'
LOG_PATH = OUTPUT_DIR / 'pipeline_domiciliations.log'
EXCEL_PATH = OUTPUT_DIR / f"domiciliations_{datetime.now().strftime('%Y%m%d_%H%M')}.xlsx"
MASTER_JSON_PATH = OUTPUT_DIR / 'domiciliations_master.json'
PIPELINE_VERSION = 'DOM_V3_CORRIGE'
PRORATA_MODE = 'CALENDAR_DAYS'
GENERER_MOIS_COMPLETS = True
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
JSON_DIR.mkdir(parents=True, exist_ok=True)
INPUT_DIR.mkdir(parents=True, exist_ok=True)
pdfs = sorted(INPUT_DIR.glob('*.pdf'))
print(f'Device          : {DEVICE}')
print(f'PDFs détectés   : {len(pdfs)}')
print(f'Entrée          : {INPUT_DIR}')
print(f'Sortie          : {OUTPUT_DIR}')
print(f'Prorata retenu  : {PRORATA_MODE}')

## 4. Chargement du modèle Qwen2.5-VL-7B

In [ ]:

if DEVICE != "cuda":
    raise RuntimeError("Ce pipeline nécessite un GPU CUDA.")

torch.backends.cuda.matmul.allow_tf32 = True
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

print("Chargement du processor...")
t0 = time.time()
processor = AutoProcessor.from_pretrained(
    MODEL_PATH,
    trust_remote_code=True,
)

print("Chargement du modèle...")
model = AutoModelForVision2Seq.from_pretrained(
    MODEL_PATH,
    torch_dtype=DTYPE,
    trust_remote_code=True,
    low_cpu_mem_usage=True,
)
model.eval()
model.to(DEVICE)

print(f"✅ Modèle chargé en {time.time() - t0:.1f}s | dtype={DTYPE}")
print(f"VRAM allouée : {torch.cuda.memory_allocated() / 1e9:.2f} GB")


## 5. Utilitaires PDF, image et JSON

In [ ]:

def sha256_file(path, chunk_size=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()


def resize_image(img, max_side=IMAGE_MAX_SIZE):
    w, h = img.size
    if max(w, h) <= max_side:
        return img
    ratio = max_side / max(w, h)
    return img.resize((int(w * ratio), int(h * ratio)), Image.LANCZOS)


def white_ratio(image):
    arr = np.array(image.convert("L"))
    return float((arr > 245).sum() / arr.size)


def is_blank(image, threshold=BLANK_THRESHOLD):
    return white_ratio(image) >= threshold


def pdf_to_pages(path, zoom=PDF_ZOOM):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"PDF introuvable : {path}")
    if path.stat().st_size == 0:
        raise ValueError(f"PDF vide : {path}")

    pages = []
    doc = fitz.open(str(path))
    try:
        page_count = int(doc.page_count)
        if page_count <= 0:
            raise ValueError(f"PyMuPDF ne détecte aucune page dans : {path.name}")

        matrix = fitz.Matrix(zoom, zoom)
        for i in range(page_count):
            page = doc.load_page(i)
            pix = page.get_pixmap(matrix=matrix, alpha=False)

            if pix.width <= 0 or pix.height <= 0 or not pix.samples:
                raise ValueError(
                    f"Rendu image vide : {path.name}, page {i + 1}"
                )

            img = Image.frombytes(
                "RGB",
                (pix.width, pix.height),
                pix.samples
            )
            img = resize_image(img)

            pages.append({
                "index": i,
                "page_num": i + 1,
                "image": img,
                "width": img.width,
                "height": img.height,
                "white_ratio": round(white_ratio(img), 6),
            })
    finally:
        doc.close()

    if len(pages) != page_count:
        raise RuntimeError(
            f"Conversion incomplète de {path.name}: "
            f"{len(pages)} image(s) pour {page_count} page(s)"
        )
    return pages


def parse_json_response(text):
    if not text:
        return {}

    clean = str(text).strip()
    clean = re.sub(r"^```(?:json)?", "", clean, flags=re.I).strip()
    clean = re.sub(r"```$", "", clean).strip()

    match = re.search(r"\{.*\}", clean, flags=re.S)
    if not match:
        return {}

    candidate = match.group(0)
    attempts = [
        candidate,
        re.sub(r",\s*([}\]])", r"\1", candidate),
    ]

    for attempt in attempts:
        try:
            parsed = json.loads(attempt)
            return parsed if isinstance(parsed, dict) else {}
        except Exception:
            continue
    return {}


def log(message):
    line = f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - {message}"
    print(line)
    with open(LOG_PATH, "a", encoding="utf-8") as f:
        f.write(line + "\n")


print("✅ Utilitaires PDF/JSON OK")


## 6. Inférence GPU batch

In [ ]:

def ask_single(prompt, image, max_new_tokens):
    messages = [{
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": prompt},
        ],
    }]

    text_in = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    inputs = processor(
        text=[text_in],
        images=[image],
        return_tensors="pt",
    ).to(DEVICE)

    t0 = time.time()
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            top_p=1.0,
            repetition_penalty=1.0,
            pad_token_id=processor.tokenizer.eos_token_id,
        )

    generated = out[0][inputs["input_ids"].shape[1]:]
    text = processor.decode(
        generated,
        skip_special_tokens=True,
        clean_up_tokenization_spaces=True,
    )

    return {
        "text": text,
        "tokens_in": int(inputs["input_ids"].shape[1]),
        "tokens_out": int(len(generated)),
        "elapsed_s": round(time.time() - t0, 3),
    }


def ask_batch(prompt, images, max_new_tokens):
    if not images:
        return []
    if len(images) == 1:
        return [ask_single(prompt, images[0], max_new_tokens)]

    texts_in = []
    for image in images:
        messages = [{
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": prompt},
            ],
        }]
        texts_in.append(
            processor.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
        )

    inputs = processor(
        text=texts_in,
        images=images,
        return_tensors="pt",
        padding=True,
    ).to(DEVICE)

    t0 = time.time()
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            temperature=1.0,
            top_p=1.0,
            repetition_penalty=1.0,
            pad_token_id=processor.tokenizer.eos_token_id,
        )

    if out.shape[0] != len(images):
        raise RuntimeError(
            f"Réponses VLM incohérentes : {out.shape[0]} sortie(s) "
            f"pour {len(images)} image(s)"
        )

    elapsed = time.time() - t0
    input_width = inputs["input_ids"].shape[1]
    attention_mask = inputs.get("attention_mask")
    results = []

    for i in range(len(images)):
        generated = out[i][input_width:]
        text = processor.decode(
            generated,
            skip_special_tokens=True,
            clean_up_tokenization_spaces=True,
        )
        tokens_in = (
            int(attention_mask[i].sum().item())
            if attention_mask is not None
            else int(input_width)
        )
        results.append({
            "text": text,
            "tokens_in": tokens_in,
            "tokens_out": int(len(generated)),
            "elapsed_s": round(elapsed / len(images), 3),
        })

    return results


print("✅ Inférence single/batch OK")


## 7. Prompts de classification et d’extraction

In [ ]:
PROMPT_CLASSIFICATION = '\nClasse cette page d\'un dossier de domiciliation de salaire d\'un travailleur étranger.\nRetourne uniquement un JSON valide :\n{"type":"TYPE", "confidence":0.00}\n\nTypes autorisés :\n- ENGAGEMENT_DOMICILIATION : titre proche de "ENGAGEMENT DE DOMICILIATION CONTRAT DES SALARIES ETRANGERS".\n- CONTRAT_TRAVAIL : contrat de travail principal, souvent "CONTRAT DE TRAVAIL A DUREE DETERMINEE".\n- CONTRAT_SPECIFIQUE : titre proche de "CONTRAT DE TRAVAIL SPECIFIQUE A LA MAIN D\'OEUVRE ETRANGERE".\n- PERMIS_TRAVAIL_DETAIL : page du permis/titre de travail contenant identité, photo, employeur, poste, dates ou validité.\n- PERMIS_TRAVAIL_COUVERTURE : couverture ou verso portant principalement "Permis de Travail", numéro de série ou extrait de loi.\n- AUTRE : page blanche, courrier sans rapport, ou document non reconnu.\n\nNe te base jamais uniquement sur le numéro de page.\n'
COMMON_RULES = "\nRègles obligatoires :\n1. Retourne uniquement un objet JSON valide, sans commentaire ni Markdown.\n2. N'invente jamais une valeur. Utilise null si absente ou illisible.\n3. Conserve une valeur brute dans les champs se terminant par _raw.\n4. Dates normalisées en YYYY-MM-DD lorsque lisibles.\n5. Montants normalisés comme nombres décimaux sans séparateur de milliers.\n6. Booléens : true, false ou null.\n7. Pour le père et la mère, conserve toujours le texte complet dans *_nom_prenom_raw.\n8. Sépare nom et prénom uniquement lorsque l'ordre est explicite ou suffisamment certain. Sinon laisse les composants null.\n9. Une valeur manuscrite ou un cachet peu lisible doit rester brute, sans reconstruction imaginative.\n"
PROMPT_ENGAGEMENT = COMMON_RULES + '\nTu lis un ENGAGEMENT DE DOMICILIATION de salaire étranger.\nExtrais le maximum d\'informations et retourne exactement les clés suivantes :\n{\n  "type":"ENGAGEMENT_DOMICILIATION",\n  "agence_domiciliataire":null,\n  "code_agence":null,\n  "client_nom_complet_raw":null,\n  "client_nom":null,\n  "client_prenom":null,\n  "compte_local_raw":null,\n  "compte_local":null,\n  "client_adresse":null,\n  "numero_contrat_raw":null,\n  "numero_contrat":null,\n  "duree_contrat_raw":null,\n  "duree_contrat_mois":null,\n  "date_debut_contrat":null,\n  "date_fin_contrat":null,\n  "employeur_raison_sociale_raw":null,\n  "employeur_raison_sociale":null,\n  "employeur_adresse":null,\n  "salaire_net_mensuel_raw":null,\n  "salaire_net_mensuel":null,\n  "part_transferable_mensuelle_raw":null,\n  "part_transferable_mensuelle":null,\n  "taux_transferable_raw":null,\n  "taux_transferable_pourcent":null,\n  "montant_total_domicilie_raw":null,\n  "montant_total_domicilie":null,\n  "devise":null,\n  "date_signature_client":null,\n  "mention_lu_et_approuve_presente":null,\n  "signature_client_presente":null,\n  "cachet_banque_present":null,\n  "date_reception_banque":null,\n  "date_domiciliation_banque":null,\n  "reference_domiciliation_raw":null,\n  "reference_domiciliation":null,\n  "agent_banque_nom":null,\n  "signature_banque_verifiee":null,\n  "confidence_globale":null\n}\n'
PROMPT_CONTRAT = COMMON_RULES + '\nTu lis un CONTRAT DE TRAVAIL d\'un travailleur étranger.\nExtrais le maximum d\'informations, notamment les informations KYC du père et de la mère.\nRetourne exactement :\n{\n  "type":"CONTRAT_TRAVAIL",\n  "reference_document_raw":null,\n  "numero_contrat_raw":null,\n  "numero_contrat":null,\n  "type_contrat":null,\n  "duree_contrat_raw":null,\n  "duree_contrat_mois":null,\n  "date_debut_contrat":null,\n  "date_fin_contrat":null,\n  "date_signature_contrat":null,\n  "travailleur_nom_complet_raw":null,\n  "travailleur_nom":null,\n  "travailleur_prenom":null,\n  "pere_nom_prenom_raw":null,\n  "pere_nom":null,\n  "pere_prenom":null,\n  "mere_nom_prenom_raw":null,\n  "mere_nom":null,\n  "mere_prenom":null,\n  "date_naissance":null,\n  "lieu_naissance":null,\n  "pays_naissance":null,\n  "nationalite":null,\n  "adresse_algerie":null,\n  "date_entree_algerie":null,\n  "poste":null,\n  "qualification_professionnelle":null,\n  "lieu_travail":null,\n  "nature_activite_employeur":null,\n  "employeur_raison_sociale_raw":null,\n  "employeur_raison_sociale":null,\n  "employeur_adresse":null,\n  "employeur_numero_adherent":null,\n  "employeur_capital_social_raw":null,\n  "employeur_capital_social":null,\n  "employeur_telephone_1":null,\n  "employeur_telephone_2":null,\n  "employeur_email":null,\n  "employeur_site_web":null,\n  "signataire_employeur_nom":null,\n  "signataire_employeur_fonction":null,\n  "salaire_brut_mensuel_raw":null,\n  "salaire_brut_mensuel":null,\n  "salaire_net_mensuel_raw":null,\n  "salaire_net_mensuel":null,\n  "affiliation_securite_sociale":null,\n  "numero_securite_sociale_algerie":null,\n  "numero_securite_sociale_pays_origine":null,\n  "titre_travail_type":null,\n  "titre_travail_numero_raw":null,\n  "titre_travail_numero":null,\n  "titre_travail_date_delivrance":null,\n  "titre_travail_date_debut":null,\n  "titre_travail_date_fin":null,\n  "signature_travailleur_presente":null,\n  "signature_employeur_presente":null,\n  "cachet_employeur_present":null,\n  "cachet_domiciliation_present":null,\n  "reference_domiciliation_raw":null,\n  "reference_domiciliation":null,\n  "date_domiciliation_banque":null,\n  "confidence_globale":null\n}\n'
PROMPT_CONTRAT_SPECIFIQUE = COMMON_RULES + '\nTu lis un CONTRAT DE TRAVAIL SPECIFIQUE A LA MAIN D\'OEUVRE ETRANGERE.\nRetourne exactement :\n{\n  "type":"CONTRAT_SPECIFIQUE",\n  "reference_document_raw":null,\n  "travailleur_nom_complet_raw":null,\n  "travailleur_nom":null,\n  "travailleur_prenom":null,\n  "pere_nom_prenom_raw":null,\n  "pere_nom":null,\n  "pere_prenom":null,\n  "mere_nom_prenom_raw":null,\n  "mere_nom":null,\n  "mere_prenom":null,\n  "date_naissance":null,\n  "lieu_naissance":null,\n  "pays_naissance":null,\n  "nationalite":null,\n  "adresse_algerie":null,\n  "date_debut_contrat":null,\n  "date_fin_contrat":null,\n  "duree_contrat_mois":null,\n  "poste":null,\n  "qualification_professionnelle":null,\n  "employeur_raison_sociale_raw":null,\n  "employeur_raison_sociale":null,\n  "employeur_adresse":null,\n  "employeur_activite":null,\n  "signataire_employeur_nom":null,\n  "signataire_employeur_fonction":null,\n  "salaire_net_mensuel_raw":null,\n  "salaire_net_mensuel":null,\n  "part_transferable_mensuelle_raw":null,\n  "part_transferable_mensuelle":null,\n  "part_payable_dzd_raw":null,\n  "part_payable_dzd":null,\n  "taux_transferable_pourcent":null,\n  "numero_securite_sociale_algerie":null,\n  "numero_securite_sociale_pays_origine":null,\n  "titre_travail_numero_raw":null,\n  "titre_travail_numero":null,\n  "titre_travail_date_delivrance":null,\n  "titre_travail_date_debut":null,\n  "titre_travail_date_fin":null,\n  "date_visa_autorite":null,\n  "numero_visa_autorite_raw":null,\n  "grade_validateur_raw":null,\n  "signature_travailleur_presente":null,\n  "signature_employeur_presente":null,\n  "cachet_employeur_present":null,\n  "visa_autorite_present":null,\n  "confidence_globale":null\n}\n'
PROMPT_PERMIS_DETAIL = COMMON_RULES + '\nTu lis la page détaillée d\'un PERMIS ou TITRE DE TRAVAIL.\nLe document peut être bilingue français/arabe et contenir une photo.\nRetourne exactement :\n{\n  "type":"PERMIS_TRAVAIL_DETAIL",\n  "permis_type":null,\n  "numero_serie_raw":null,\n  "numero_serie":null,\n  "numero_permis_raw":null,\n  "numero_permis":null,\n  "reference_document_raw":null,\n  "travailleur_nom_complet_raw":null,\n  "travailleur_nom":null,\n  "travailleur_prenom":null,\n  "pere_nom_prenom_raw":null,\n  "pere_nom":null,\n  "pere_prenom":null,\n  "mere_nom_prenom_raw":null,\n  "mere_nom":null,\n  "mere_prenom":null,\n  "date_naissance":null,\n  "lieu_naissance":null,\n  "pays_naissance":null,\n  "nationalite":null,\n  "date_entree_algerie":null,\n  "poste":null,\n  "qualification_professionnelle":null,\n  "date_delivrance":null,\n  "lieu_delivrance":null,\n  "date_debut_validite":null,\n  "date_fin_validite":null,\n  "duree_raw":null,\n  "duree_mois":null,\n  "employeur_raison_sociale_raw":null,\n  "employeur_raison_sociale":null,\n  "employeur_activite":null,\n  "employeur_adresse":null,\n  "photo_presente":null,\n  "cachet_autorite_present":null,\n  "signature_autorite_presente":null,\n  "confidence_globale":null\n}\n'
PROMPT_PERMIS_COUVERTURE = COMMON_RULES + '\nTu lis la couverture ou le verso d\'un PERMIS DE TRAVAIL.\nRetourne exactement :\n{\n  "type":"PERMIS_TRAVAIL_COUVERTURE",\n  "titre_document":null,\n  "numero_serie_raw":null,\n  "numero_serie":null,\n  "wilaya_delivrance":null,\n  "ministere":null,\n  "texte_legal_reference":null,\n  "cachet_autorite_present":null,\n  "confidence_globale":null\n}\n'
PROMPTS_EXTRACTION = {'ENGAGEMENT_DOMICILIATION': PROMPT_ENGAGEMENT, 'CONTRAT_TRAVAIL': PROMPT_CONTRAT, 'CONTRAT_SPECIFIQUE': PROMPT_CONTRAT_SPECIFIQUE, 'PERMIS_TRAVAIL_DETAIL': PROMPT_PERMIS_DETAIL, 'PERMIS_TRAVAIL_COUVERTURE': PROMPT_PERMIS_COUVERTURE}
TYPES_VALIDES = set(PROMPTS_EXTRACTION) | {'AUTRE'}
print('✅ Prompts spécialisés chargés')

## 8. Normalisation technique

In [ ]:
NULL_VALUES = {'', 'NULL', 'NONE', 'N/A', 'NA', 'NEANT', 'NÉANT', 'ILLISIBLE'}

def norm_text(value):
    if value is None:
        return None
    text = re.sub('\\s+', ' ', str(value).replace('\xa0', ' ').strip())
    return None if text.upper() in NULL_VALUES else text

def norm_upper(value):
    text = norm_text(value)
    return text.upper() if text else None

def norm_digits(value):
    text = norm_text(value)
    return re.sub('\\D', '', text) if text else None

def norm_alnum(value):
    text = norm_text(value)
    return re.sub('[^A-Z0-9]', '', text.upper()) if text else None

def norm_reference(value):
    text = norm_upper(value)
    if not text:
        return None
    text = re.sub('\\s+', '', text)
    text = re.sub('[^A-Z0-9./-]', '', text)
    return text or None

def norm_amount(value):
    if value is None:
        return None
    if isinstance(value, bool):
        return None
    if isinstance(value, (int, float)):
        return round(float(value), 2)
    s = str(value).strip().replace('\xa0', ' ')
    s = re.sub('[^0-9,.-]', '', s)
    if not s or s in {'-', '.', ','}:
        return None
    last_comma = s.rfind(',')
    last_dot = s.rfind('.')
    last_sep = max(last_comma, last_dot)
    decimal_sep = None
    if last_sep >= 0 and 1 <= len(s) - last_sep - 1 <= 2:
        decimal_sep = s[last_sep]
    if decimal_sep:
        integer = re.sub('[.,]', '', s[:last_sep])
        decimal = re.sub('[.,]', '', s[last_sep + 1:])
        normalized = f'{integer}.{decimal}'
    else:
        normalized = re.sub('[.,]', '', s)
    try:
        return round(float(normalized), 2)
    except ValueError:
        return None

def norm_percent(value):
    amount = norm_amount(value)
    if amount is None:
        return None
    if 0 < amount <= 1:
        amount *= 100
    return round(amount, 4)

def norm_bool(value):
    if value is None or isinstance(value, bool):
        return value
    text = norm_upper(value)
    if text in {'OUI', 'TRUE', 'VRAI', 'PRESENT', 'PRÉSENT', '1'}:
        return True
    if text in {'NON', 'FALSE', 'FAUX', 'ABSENT', '0'}:
        return False
    return None
DATE_FORMATS = ['%Y-%m-%d', '%d/%m/%Y', '%d-%m-%Y', '%d.%m.%Y', '%Y/%m/%d', '%d/%m/%y', '%d-%m-%y']

def parse_date_any(value):
    text = norm_text(value)
    if not text:
        return None
    text = text.replace(' ', '')
    for fmt in DATE_FORMATS:
        try:
            return datetime.strptime(text, fmt).date()
        except ValueError:
            pass
    return None

def norm_date(value):
    parsed = parse_date_any(value)
    return parsed.isoformat() if parsed else norm_text(value)
AMOUNT_FIELDS = {'salaire_net_mensuel', 'salaire_brut_mensuel', 'part_transferable_mensuelle', 'part_payable_dzd', 'montant_total_domicilie', 'employeur_capital_social'}
PERCENT_FIELDS = {'taux_transferable_pourcent'}
DATE_FIELDS = {'date_debut_contrat', 'date_fin_contrat', 'date_signature_contrat', 'date_naissance', 'date_entree_algerie', 'titre_travail_date_delivrance', 'titre_travail_date_debut', 'titre_travail_date_fin', 'date_visa_autorite', 'date_delivrance', 'date_debut_validite', 'date_fin_validite', 'date_signature_client', 'date_reception_banque', 'date_domiciliation_banque'}
BOOL_SUFFIXES = ('_presente', '_present', '_verifiee')
DIGIT_FIELDS = {'compte_local', 'numero_securite_sociale_algerie', 'numero_securite_sociale_pays_origine', 'employeur_numero_adherent'}
REFERENCE_FIELDS = {'numero_contrat', 'reference_domiciliation', 'numero_serie', 'numero_permis', 'titre_travail_numero', 'reference_document'}
UPPER_FIELDS = {'client_nom', 'client_prenom', 'travailleur_nom', 'travailleur_prenom', 'pere_nom', 'pere_prenom', 'mere_nom', 'mere_prenom', 'nationalite', 'pays_naissance', 'lieu_naissance', 'poste', 'qualification_professionnelle', 'employeur_raison_sociale', 'signataire_employeur_nom', 'signataire_employeur_fonction', 'agence_domiciliataire', 'lieu_delivrance'}

def normalize_extraction(data):
    out = {}
    for key, value in (data or {}).items():
        if key in AMOUNT_FIELDS:
            out[key] = norm_amount(value)
        elif key in PERCENT_FIELDS:
            out[key] = norm_percent(value)
        elif key in DATE_FIELDS:
            out[key] = norm_date(value)
        elif key in DIGIT_FIELDS:
            out[key] = norm_digits(value)
        elif key in REFERENCE_FIELDS:
            out[key] = norm_reference(value)
        elif key in UPPER_FIELDS:
            out[key] = norm_upper(value)
        elif key.endswith(BOOL_SUFFIXES):
            out[key] = norm_bool(value)
        elif key == 'confidence_globale':
            try:
                out[key] = round(float(value), 4) if value is not None else None
            except Exception:
                out[key] = None
        elif isinstance(value, str):
            out[key] = norm_text(value)
        else:
            out[key] = value
    return out
print('✅ Normalisation OK')

## 9. Consolidation des documents et informations KYC

In [ ]:
DOC_PREFIX = {'ENGAGEMENT_DOMICILIATION': 'ENG', 'CONTRAT_TRAVAIL': 'CTR', 'CONTRAT_SPECIFIQUE': 'CTS', 'PERMIS_TRAVAIL_DETAIL': 'PTR', 'PERMIS_TRAVAIL_COUVERTURE': 'PTC'}

def first_value(*values):
    for value in values:
        if value is not None and value != '':
            return value
    return None

def merge_occurrences(occurrences):
    """Fusion technique : complète les valeurs manquantes sans masquer les conflits."""
    merged = {}
    conflicts = []
    for occurrence in occurrences:
        data = occurrence.get('normalized', {})
        for key, value in data.items():
            if value is None:
                continue
            if key not in merged or merged[key] is None:
                merged[key] = value
            elif merged[key] != value:
                conflicts.append({'field': key, 'kept': merged[key], 'other': value, 'page': occurrence.get('page_num')})
    return (merged, conflicts)

def flatten_dict(data, prefix='', sep='_'):
    out = {}
    for key, value in data.items():
        new_key = f'{prefix}{sep}{key}' if prefix else str(key)
        if isinstance(value, dict):
            out.update(flatten_dict(value, new_key, sep=sep))
        elif isinstance(value, list):
            out[new_key] = json.dumps(value, ensure_ascii=False, default=str)
        else:
            out[new_key] = value
    return out

def compute_duration_months(start_iso, end_iso):
    """Durée en mois complets, bornes inclusives.

    Exemple : 12/06/2025 au 11/06/2026 = 12 mois.
    """
    start = parse_date_any(start_iso)
    end = parse_date_any(end_iso)
    if not start or not end or end < start:
        return None
    end_exclusive = end + timedelta(days=1)
    months = (end_exclusive.year - start.year) * 12 + (end_exclusive.month - start.month)
    if end_exclusive.day < start.day:
        months -= 1
    return max(months, 0)

def consolidate_dossier(pdf_path, page_records, stats):
    by_type = defaultdict(list)
    for record in page_records:
        doc_type = record.get('doc_type')
        if doc_type in DOC_PREFIX and record.get('normalized'):
            by_type[doc_type].append(record)
    merged_docs = {}
    all_conflicts = []
    for doc_type, occurrences in by_type.items():
        merged, conflicts = merge_occurrences(occurrences)
        merged_docs[doc_type] = merged
        for conflict in conflicts:
            conflict['doc_type'] = doc_type
        all_conflicts.extend(conflicts)
    eng = merged_docs.get('ENGAGEMENT_DOMICILIATION', {})
    ctr = merged_docs.get('CONTRAT_TRAVAIL', {})
    cts = merged_docs.get('CONTRAT_SPECIFIQUE', {})
    ptr = merged_docs.get('PERMIS_TRAVAIL_DETAIL', {})
    ptc = merged_docs.get('PERMIS_TRAVAIL_COUVERTURE', {})
    can = {'DOM_ID': first_value(eng.get('reference_domiciliation'), ctr.get('reference_domiciliation')), 'DOM_ID_RAW': first_value(eng.get('reference_domiciliation_raw'), ctr.get('reference_domiciliation_raw')), 'DOM_AGENCE': eng.get('agence_domiciliataire'), 'DOM_CODE_AGENCE': first_value(eng.get('code_agence')), 'DOM_DATE_SIGNATURE_CLIENT': eng.get('date_signature_client'), 'DOM_DATE_RECEPTION_BANQUE': eng.get('date_reception_banque'), 'DOM_DATE_BANQUE': first_value(eng.get('date_domiciliation_banque'), ctr.get('date_domiciliation_banque')), 'DOM_AGENT_BANQUE': eng.get('agent_banque_nom'), 'KYC_NOM_COMPLET_RAW': first_value(eng.get('client_nom_complet_raw'), ctr.get('travailleur_nom_complet_raw'), cts.get('travailleur_nom_complet_raw'), ptr.get('travailleur_nom_complet_raw')), 'KYC_NOM': first_value(eng.get('client_nom'), ctr.get('travailleur_nom'), cts.get('travailleur_nom'), ptr.get('travailleur_nom')), 'KYC_PRENOM': first_value(eng.get('client_prenom'), ctr.get('travailleur_prenom'), cts.get('travailleur_prenom'), ptr.get('travailleur_prenom')), 'KYC_PERE_NOM_PRENOM_RAW': first_value(ctr.get('pere_nom_prenom_raw'), cts.get('pere_nom_prenom_raw'), ptr.get('pere_nom_prenom_raw')), 'KYC_PERE_NOM': first_value(ctr.get('pere_nom'), cts.get('pere_nom'), ptr.get('pere_nom')), 'KYC_PERE_PRENOM': first_value(ctr.get('pere_prenom'), cts.get('pere_prenom'), ptr.get('pere_prenom')), 'KYC_MERE_NOM_PRENOM_RAW': first_value(ctr.get('mere_nom_prenom_raw'), cts.get('mere_nom_prenom_raw'), ptr.get('mere_nom_prenom_raw')), 'KYC_MERE_NOM': first_value(ctr.get('mere_nom'), cts.get('mere_nom'), ptr.get('mere_nom')), 'KYC_MERE_PRENOM': first_value(ctr.get('mere_prenom'), cts.get('mere_prenom'), ptr.get('mere_prenom')), 'KYC_DATE_NAISSANCE': first_value(ctr.get('date_naissance'), cts.get('date_naissance'), ptr.get('date_naissance')), 'KYC_LIEU_NAISSANCE': first_value(ctr.get('lieu_naissance'), cts.get('lieu_naissance'), ptr.get('lieu_naissance')), 'KYC_PAYS_NAISSANCE': first_value(ctr.get('pays_naissance'), cts.get('pays_naissance'), ptr.get('pays_naissance')), 'KYC_NATIONALITE': first_value(ctr.get('nationalite'), cts.get('nationalite'), ptr.get('nationalite')), 'KYC_ADRESSE_ALGERIE': first_value(ctr.get('adresse_algerie'), cts.get('adresse_algerie'), eng.get('client_adresse')), 'KYC_DATE_ENTREE_ALGERIE': first_value(ctr.get('date_entree_algerie'), ptr.get('date_entree_algerie')), 'CPT_COMPTE_LOCAL_RAW': eng.get('compte_local_raw'), 'CPT_COMPTE_LOCAL': eng.get('compte_local'), 'EMP_RAISON_SOCIALE_RAW': first_value(eng.get('employeur_raison_sociale_raw'), ctr.get('employeur_raison_sociale_raw'), cts.get('employeur_raison_sociale_raw'), ptr.get('employeur_raison_sociale_raw')), 'EMP_RAISON_SOCIALE': first_value(eng.get('employeur_raison_sociale'), ctr.get('employeur_raison_sociale'), cts.get('employeur_raison_sociale'), ptr.get('employeur_raison_sociale')), 'EMP_ADRESSE': first_value(eng.get('employeur_adresse'), ctr.get('employeur_adresse'), cts.get('employeur_adresse'), ptr.get('employeur_adresse')), 'EMP_ACTIVITE': first_value(ctr.get('nature_activite_employeur'), cts.get('employeur_activite'), ptr.get('employeur_activite')), 'EMP_NUMERO_ADHERENT': ctr.get('employeur_numero_adherent'), 'EMP_CAPITAL_SOCIAL': ctr.get('employeur_capital_social'), 'EMP_TELEPHONE_1': ctr.get('employeur_telephone_1'), 'EMP_TELEPHONE_2': ctr.get('employeur_telephone_2'), 'EMP_EMAIL': ctr.get('employeur_email'), 'EMP_SITE_WEB': ctr.get('employeur_site_web'), 'CTR_NUMERO_RAW': first_value(eng.get('numero_contrat_raw'), ctr.get('numero_contrat_raw')), 'CTR_NUMERO': first_value(eng.get('numero_contrat'), ctr.get('numero_contrat')), 'CTR_REFERENCE_DOCUMENT': first_value(ctr.get('reference_document_raw'), cts.get('reference_document_raw')), 'CTR_TYPE': first_value(ctr.get('type_contrat'), 'CDD' if ctr else None), 'CTR_DATE_DEBUT': first_value(eng.get('date_debut_contrat'), ctr.get('date_debut_contrat'), cts.get('date_debut_contrat'), ptr.get('date_debut_validite')), 'CTR_DATE_FIN': first_value(eng.get('date_fin_contrat'), ctr.get('date_fin_contrat'), cts.get('date_fin_contrat'), ptr.get('date_fin_validite')), 'CTR_DUREE_MOIS': first_value(eng.get('duree_contrat_mois'), ctr.get('duree_contrat_mois'), cts.get('duree_contrat_mois'), ptr.get('duree_mois')), 'CTR_POSTE': first_value(ctr.get('poste'), cts.get('poste'), ptr.get('poste')), 'CTR_QUALIFICATION': first_value(ctr.get('qualification_professionnelle'), cts.get('qualification_professionnelle'), ptr.get('qualification_professionnelle')), 'CTR_LIEU_TRAVAIL': ctr.get('lieu_travail'), 'REM_SALAIRE_BRUT_MENSUEL': ctr.get('salaire_brut_mensuel'), 'REM_SALAIRE_NET_MENSUEL': first_value(eng.get('salaire_net_mensuel'), ctr.get('salaire_net_mensuel'), cts.get('salaire_net_mensuel')), 'REM_PART_TRANSFERABLE_MENSUELLE': first_value(eng.get('part_transferable_mensuelle'), cts.get('part_transferable_mensuelle')), 'REM_TAUX_TRANSFERABLE_PCT': first_value(eng.get('taux_transferable_pourcent'), cts.get('taux_transferable_pourcent')), 'REM_PART_PAYABLE_DZD': cts.get('part_payable_dzd'), 'REM_MONTANT_TOTAL_DOMICILIE': eng.get('montant_total_domicilie'), 'REM_DEVISE': first_value(eng.get('devise'), 'DZD'), 'SS_AFFILIATION': ctr.get('affiliation_securite_sociale'), 'SS_NUMERO_ALGERIE': first_value(ctr.get('numero_securite_sociale_algerie'), cts.get('numero_securite_sociale_algerie')), 'SS_NUMERO_PAYS_ORIGINE': first_value(ctr.get('numero_securite_sociale_pays_origine'), cts.get('numero_securite_sociale_pays_origine')), 'PTR_TYPE': first_value(ptr.get('permis_type'), ctr.get('titre_travail_type')), 'PTR_NUMERO_RAW': first_value(ptr.get('numero_permis_raw'), ctr.get('titre_travail_numero_raw'), cts.get('titre_travail_numero_raw')), 'PTR_NUMERO': first_value(ptr.get('numero_permis'), ctr.get('titre_travail_numero'), cts.get('titre_travail_numero')), 'PTR_NUMERO_SERIE_RAW': first_value(ptr.get('numero_serie_raw'), ptc.get('numero_serie_raw')), 'PTR_NUMERO_SERIE': first_value(ptr.get('numero_serie'), ptc.get('numero_serie')), 'PTR_DATE_DELIVRANCE': first_value(ptr.get('date_delivrance'), ctr.get('titre_travail_date_delivrance'), cts.get('titre_travail_date_delivrance')), 'PTR_LIEU_DELIVRANCE': first_value(ptr.get('lieu_delivrance'), ptc.get('wilaya_delivrance')), 'PTR_DATE_DEBUT': first_value(ptr.get('date_debut_validite'), ctr.get('titre_travail_date_debut'), cts.get('titre_travail_date_debut')), 'PTR_DATE_FIN': first_value(ptr.get('date_fin_validite'), ctr.get('titre_travail_date_fin'), cts.get('titre_travail_date_fin')), 'SIG_TRAVAILLEUR_PRESENTE': first_value(ctr.get('signature_travailleur_presente'), cts.get('signature_travailleur_presente')), 'SIG_EMPLOYEUR_PRESENTE': first_value(ctr.get('signature_employeur_presente'), cts.get('signature_employeur_presente')), 'SIG_CACHET_EMPLOYEUR_PRESENT': first_value(ctr.get('cachet_employeur_present'), cts.get('cachet_employeur_present')), 'SIG_CACHET_AUTORITE_PRESENT': first_value(ptr.get('cachet_autorite_present'), ptc.get('cachet_autorite_present'))}
    can['DOC_ENGAGEMENT_PRESENT'] = bool(eng)
    can['DOC_CONTRAT_TRAVAIL_PRESENT'] = bool(ctr)
    can['DOC_CONTRAT_SPECIFIQUE_PRESENT'] = bool(cts)
    can['DOC_PERMIS_DETAIL_PRESENT'] = bool(ptr)
    can['DOC_PERMIS_COUVERTURE_PRESENT'] = bool(ptc)
    can['DOC_NOMBRE_TYPES_DETECTES'] = len(by_type)
    if can['CTR_DUREE_MOIS'] is None:
        can['CTR_DUREE_MOIS'] = compute_duration_months(can['CTR_DATE_DEBUT'], can['CTR_DATE_FIN'])
    can['SRC_FICHIER'] = pdf_path.name
    can['SRC_SHA256'] = sha256_file(pdf_path)
    can['SRC_NOMBRE_PAGES'] = len(page_records)
    can['SRC_DATE_EXTRACTION'] = datetime.now().isoformat(timespec='seconds')
    can['SRC_MODELE'] = 'Qwen2.5-VL-7B-Instruct'
    can['SRC_VERSION_PIPELINE'] = PIPELINE_VERSION
    can['QLT_TOKENS_IN'] = stats.get('tokens_in', 0)
    can['QLT_TOKENS_OUT'] = stats.get('tokens_out', 0)
    can['QLT_TEMPS_S'] = stats.get('elapsed_s')
    can['QLT_CONFLITS_NB'] = len(all_conflicts)
    can['QLT_CONFLITS_JSON'] = json.dumps(all_conflicts, ensure_ascii=False) if all_conflicts else None
    can['QLT_TYPES_DOCUMENTS'] = ', '.join(sorted(by_type))
    can['QLT_REVUE_REQUISE'] = bool(all_conflicts) or not can.get('CTR_DATE_DEBUT') or (not can.get('CTR_DATE_FIN'))
    flat = dict(can)
    for doc_type, merged in merged_docs.items():
        prefix = DOC_PREFIX[doc_type]
        for key, value in merged.items():
            flat[f'{prefix}_{key.upper()}'] = value
    return {'canonical': can, 'flat': flat, 'documents_merged': merged_docs, 'page_records': page_records, 'conflicts': all_conflicts, 'stats': stats}
print('✅ Consolidation KYC et domiciliation OK')

## 10. Gestion des mois scindés P1 / P2 et du planning TL

In [ ]:
def month_start(d):
    return d.replace(day=1)

def month_last_day(d):
    return d.replace(day=calendar.monthrange(d.year, d.month)[1])

def iter_month_starts(start, end):
    current = month_start(start)
    last = month_start(end)
    while current <= last:
        yield current
        if current.month == 12:
            current = date(current.year + 1, 1, 1)
        else:
            current = date(current.year, current.month + 1, 1)

def segment_part(seg_start, seg_end, m_start, m_end):
    if seg_start == m_start and seg_end == m_end:
        return 'FULL'
    if seg_start == m_start and seg_end < m_end:
        return 'P1'
    if seg_start > m_start and seg_end == m_end:
        return 'P2'
    return 'PX'

def segment_label(month, part):
    base = month.strftime('%Y-%m')
    return base if part == 'FULL' else f'{base}{part}'

def prorata_amounts(monthly_cap, active_days, days_in_month, part):
    cap = norm_amount(monthly_cap)
    if cap is None:
        return {'calendar': None, 'thirty': None, 'selected': None}
    if part == 'FULL':
        calendar_amount = cap
        thirty_amount = cap
    else:
        calendar_amount = round(cap * active_days / days_in_month, 2)
        thirty_amount = round(cap * active_days / 30, 2)
    selected = calendar_amount if PRORATA_MODE == 'CALENDAR_DAYS' else thirty_amount
    return {'calendar': calendar_amount, 'thirty': thirty_amount, 'selected': selected}

def build_contract_segments(dom_flat):
    start = parse_date_any(dom_flat.get('CTR_DATE_DEBUT'))
    end = parse_date_any(dom_flat.get('CTR_DATE_FIN'))
    if not start or not end or end < start:
        return []
    monthly_cap = dom_flat.get('REM_PART_TRANSFERABLE_MENSUELLE')
    today = date.today()
    rows = []
    for m_start in iter_month_starts(start, end):
        m_end = month_last_day(m_start)
        seg_start = max(start, m_start)
        seg_end = min(end, m_end)
        active_days = (seg_end - seg_start).days + 1
        days_in_month = m_end.day
        part = segment_part(seg_start, seg_end, m_start, m_end)
        amounts = prorata_amounts(monthly_cap, active_days, days_in_month, part)
        if m_end < today:
            temporal_status = 'PASSEE'
        elif m_start > today:
            temporal_status = 'FUTURE'
        else:
            temporal_status = 'EN_COURS'
        rows.append({'PLN_DOM_ID': dom_flat.get('DOM_ID'), 'PLN_FICHIER_SOURCE': dom_flat.get('SRC_FICHIER'), 'PLN_COMPTE_LOCAL': dom_flat.get('CPT_COMPTE_LOCAL'), 'PLN_NOM': dom_flat.get('KYC_NOM'), 'PLN_PRENOM': dom_flat.get('KYC_PRENOM'), 'PLN_CONTRAT_NUMERO': dom_flat.get('CTR_NUMERO'), 'PLN_CONTRAT_DATE_DEBUT': dom_flat.get('CTR_DATE_DEBUT'), 'PLN_CONTRAT_DATE_FIN': dom_flat.get('CTR_DATE_FIN'), 'PLN_PERIODE': segment_label(m_start, part), 'PLN_MOIS_BASE': m_start.strftime('%Y-%m'), 'PLN_PARTIE': part, 'PLN_DATE_DEBUT_SEGMENT': seg_start.isoformat(), 'PLN_DATE_FIN_SEGMENT': seg_end.isoformat(), 'PLN_NB_JOURS_SEGMENT': active_days, 'PLN_NB_JOURS_MOIS': days_in_month, 'PLN_TAUX_PRORATA_CALENDAIRE': round(active_days / days_in_month, 8), 'PLN_TAUX_PRORATA_30J': round(active_days / 30, 8) if part != 'FULL' else 1.0, 'PLN_PLAFOND_MENSUEL_CONTRACTUEL': monthly_cap, 'PLN_MONTANT_THEORIQUE_CALENDAIRE': amounts['calendar'], 'PLN_MONTANT_THEORIQUE_30J': amounts['thirty'], 'PLN_MONTANT_REFERENCE': amounts['selected'], 'PLN_MODE_PRORATA_REFERENCE': PRORATA_MODE, 'PLN_MONTANT_AUTORISE_SAISI': None, 'PLN_DATE_VALIDATION': None, 'PLN_VALIDATEUR': None, 'PLN_COMMENTAIRE': None, 'PLN_STATUT_TEMPOREL': temporal_status})
    return rows

def add_boundary_amounts_to_dom(dom_flat, planning_rows):
    out = dict(dom_flat)
    p1 = next((r for r in planning_rows if r['PLN_PARTIE'] == 'P1'), None)
    p2 = next((r for r in planning_rows if r['PLN_PARTIE'] == 'P2'), None)
    for label, row in (('P1', p1), ('P2', p2)):
        out[f'DOM_{label}_PERIODE'] = row.get('PLN_PERIODE') if row else None
        out[f'DOM_{label}_DATE_DEBUT'] = row.get('PLN_DATE_DEBUT_SEGMENT') if row else None
        out[f'DOM_{label}_DATE_FIN'] = row.get('PLN_DATE_FIN_SEGMENT') if row else None
        out[f'DOM_{label}_NB_JOURS'] = row.get('PLN_NB_JOURS_SEGMENT') if row else None
        out[f'DOM_{label}_MONTANT_CALENDAIRE'] = row.get('PLN_MONTANT_THEORIQUE_CALENDAIRE') if row else None
        out[f'DOM_{label}_MONTANT_30J'] = row.get('PLN_MONTANT_THEORIQUE_30J') if row else None
        out[f'DOM_{label}_MONTANT_REFERENCE'] = row.get('PLN_MONTANT_REFERENCE') if row else None
    return out

def renewal_group_key(row):
    account = row.get('CPT_COMPTE_LOCAL')
    if account:
        return f'ACCOUNT:{account}'
    return 'KYC:' + '|'.join((str(row.get(k) or '') for k in ['KYC_NOM', 'KYC_PRENOM', 'KYC_DATE_NAISSANCE']))

def add_renewal_links(dom_rows):
    groups = defaultdict(list)
    for row in dom_rows:
        groups[renewal_group_key(row)].append(row)
    for group_key, rows in groups.items():
        rows.sort(key=lambda r: parse_date_any(r.get('CTR_DATE_DEBUT')) or date.max)
        for i, row in enumerate(rows):
            previous = rows[i - 1] if i > 0 else None
            following = rows[i + 1] if i + 1 < len(rows) else None
            row['REN_GROUPE'] = group_key
            row['REN_CONTRAT_PRECEDENT'] = previous.get('CTR_NUMERO') if previous else None
            row['REN_DATE_FIN_PRECEDENT'] = previous.get('CTR_DATE_FIN') if previous else None
            row['REN_CONTRAT_SUIVANT'] = following.get('CTR_NUMERO') if following else None
            row['REN_DATE_DEBUT_SUIVANT'] = following.get('CTR_DATE_DEBUT') if following else None
            if previous:
                prev_end = parse_date_any(previous.get('CTR_DATE_FIN'))
                current_start = parse_date_any(row.get('CTR_DATE_DEBUT'))
                gap = (current_start - prev_end).days - 1 if prev_end and current_start else None
                row['REN_ECART_JOURS_PRECEDENT'] = gap
                if gap == 0:
                    row['REN_STATUT_CONTINUITE'] = 'CONTIGU'
                elif gap is not None and gap > 0:
                    row['REN_STATUT_CONTINUITE'] = 'RUPTURE'
                elif gap is not None:
                    row['REN_STATUT_CONTINUITE'] = 'CHEVAUCHEMENT'
            else:
                row['REN_ECART_JOURS_PRECEDENT'] = None
                row['REN_STATUT_CONTINUITE'] = 'PREMIER_CONTRAT'
    return dom_rows
print('✅ Planning TL et renouvellements OK')

### Test de la règle P1 / P2 demandée

In [ ]:
_demo_dom = {'DOM_ID': 'DEMO', 'CTR_NUMERO': 'NOUVEAU', 'CTR_DATE_DEBUT': '2025-06-12', 'CTR_DATE_FIN': '2026-06-11', 'REM_PART_TRANSFERABLE_MENSUELLE': 442985.84}
_demo = build_contract_segments(_demo_dom)
assert _demo[0]['PLN_PERIODE'] == '2025-06P2'
assert _demo[0]['PLN_NB_JOURS_SEGMENT'] == 19
assert _demo[-1]['PLN_PERIODE'] == '2026-06P1'
assert _demo[-1]['PLN_NB_JOURS_SEGMENT'] == 11
print('✅ Test P1/P2 réussi')
print({k: _demo[0][k] for k in ['PLN_PERIODE', 'PLN_NB_JOURS_SEGMENT', 'PLN_MONTANT_REFERENCE']})
print({k: _demo[-1][k] for k in ['PLN_PERIODE', 'PLN_NB_JOURS_SEGMENT', 'PLN_MONTANT_REFERENCE']})

## 11. Classification et extraction d’un dossier

In [ ]:
def classify_pages(pages):
    active_pages = [p for p in pages if not is_blank(p['image'])]
    records = []
    for start in range(0, len(active_pages), GPU_BATCH_SIZE_CLASSIFICATION):
        batch = active_pages[start:start + GPU_BATCH_SIZE_CLASSIFICATION]
        responses = ask_batch(PROMPT_CLASSIFICATION, [p['image'] for p in batch], MAX_NEW_TOKENS_CLASSIFICATION)
        for page, response in zip(batch, responses):
            parsed = parse_json_response(response['text'])
            doc_type = norm_upper(parsed.get('type')) or 'AUTRE'
            if doc_type not in TYPES_VALIDES:
                doc_type = 'AUTRE'
            try:
                confidence = float(parsed.get('confidence')) if parsed.get('confidence') is not None else None
            except Exception:
                confidence = None
            records.append({'page_index': page['index'], 'page_num': page['page_num'], 'width': page['width'], 'height': page['height'], 'white_ratio': page['white_ratio'], 'image': page['image'], 'doc_type': doc_type, 'classification_confidence': confidence, 'classification_raw': response['text'], 'classification_tokens_in': response['tokens_in'], 'classification_tokens_out': response['tokens_out'], 'classification_elapsed_s': response['elapsed_s']})
    active_indexes = {r['page_index'] for r in records}
    for page in pages:
        if page['index'] not in active_indexes:
            records.append({'page_index': page['index'], 'page_num': page['page_num'], 'width': page['width'], 'height': page['height'], 'white_ratio': page['white_ratio'], 'image': page['image'], 'doc_type': 'AUTRE', 'classification_confidence': 1.0, 'classification_raw': 'PAGE_BLANCHE_TECHNIQUE', 'classification_tokens_in': 0, 'classification_tokens_out': 0, 'classification_elapsed_s': 0.0})
    return sorted(records, key=lambda r: r['page_index'])

def extract_classified_pages(records):
    grouped = defaultdict(list)
    for record in records:
        if record['doc_type'] in PROMPTS_EXTRACTION:
            grouped[record['doc_type']].append(record)
    for doc_type, group in grouped.items():
        prompt = PROMPTS_EXTRACTION[doc_type]
        for start in range(0, len(group), GPU_BATCH_SIZE_EXTRACTION):
            batch = group[start:start + GPU_BATCH_SIZE_EXTRACTION]
            responses = ask_batch(prompt, [r['image'] for r in batch], MAX_NEW_TOKENS_EXTRACTION)
            for record, response in zip(batch, responses):
                raw_data = parse_json_response(response['text'])
                normalized = normalize_extraction(raw_data)
                record['extraction_raw_text'] = response['text']
                record['raw_data'] = raw_data
                record['normalized'] = normalized
                record['extraction_tokens_in'] = response['tokens_in']
                record['extraction_tokens_out'] = response['tokens_out']
                record['extraction_elapsed_s'] = response['elapsed_s']
    for record in records:
        record.pop('image', None)
        record.setdefault('extraction_raw_text', None)
        record.setdefault('raw_data', {})
        record.setdefault('normalized', {})
        record.setdefault('extraction_tokens_in', 0)
        record.setdefault('extraction_tokens_out', 0)
        record.setdefault('extraction_elapsed_s', 0.0)
    return records

def process_pdf(pdf_path, verbose=True):
    t0 = time.time()
    if verbose:
        log(f'📁 {pdf_path.name}')
    pages = pdf_to_pages(pdf_path)
    if not pages:
        raise ValueError(f'Aucune page détectée dans {pdf_path.name}')
    records = classify_pages(pages)
    if not records:
        raise ValueError(f'Aucune page classifiée pour {pdf_path.name}')
    records = extract_classified_pages(records)
    if not records:
        raise ValueError(f"Aucun résultat d'extraction pour {pdf_path.name}")
    tokens_in = sum((r.get('classification_tokens_in', 0) + r.get('extraction_tokens_in', 0) for r in records))
    tokens_out = sum((r.get('classification_tokens_out', 0) + r.get('extraction_tokens_out', 0) for r in records))
    stats = {'tokens_in': tokens_in, 'tokens_out': tokens_out, 'tokens_total': tokens_in + tokens_out, 'elapsed_s': round(time.time() - t0, 3), 'pages': len(pages)}
    dossier = consolidate_dossier(pdf_path, records, stats)
    planning = build_contract_segments(dossier['flat'])
    dossier['flat'] = add_boundary_amounts_to_dom(dossier['flat'], planning)
    dossier['planning_tl'] = planning
    checkpoint = JSON_DIR / f'{pdf_path.stem}__{sha256_file(pdf_path)[:12]}.json'
    with open(checkpoint, 'w', encoding='utf-8') as f:
        json.dump(dossier, f, ensure_ascii=False, indent=2, default=str)
    if verbose:
        log(f"✅ {pdf_path.name} | pages={stats['pages']} | tokens={stats['tokens_total']} | planning={len(planning)}")
    return dossier
print('✅ Traitement dossier OK')

## 12. Export Excel compatible Alteryx

In [ ]:
PREFIX_ORDER = ['DOM_', 'DOC_', 'KYC_', 'CPT_', 'EMP_', 'CTR_', 'REM_', 'SS_', 'PTR_', 'SIG_', 'REN_', 'ENG_', 'CTS_', 'PTC_', 'SRC_', 'QLT_']

def ordered_columns(rows):
    cols = set()
    for row in rows:
        cols.update(row.keys())
    ordered = []
    remaining = set(cols)
    for prefix in PREFIX_ORDER:
        prefixed = sorted((c for c in remaining if c.startswith(prefix)))
        ordered.extend(prefixed)
        remaining -= set(prefixed)
    ordered.extend(sorted(remaining))
    return ordered

def sheet_from_rows(wb, title, rows, columns=None):
    ws = wb.create_sheet(title)
    if columns is None:
        columns = ordered_columns(rows) if rows else []
    if not columns:
        ws['A1'] = 'Aucune donnée'
        return ws
    header_fill = PatternFill('solid', fgColor='1F4E78')
    header_font = Font(color='FFFFFF', bold=True, name='Arial', size=9)
    thin = Side(style='thin', color='D9E2F3')
    for col_idx, name in enumerate(columns, start=1):
        cell = ws.cell(row=1, column=col_idx, value=name)
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
        cell.border = Border(bottom=thin, right=thin)
    for row_idx, row in enumerate(rows, start=2):
        for col_idx, name in enumerate(columns, start=1):
            value = row.get(name)
            if isinstance(value, (list, dict)):
                value = json.dumps(value, ensure_ascii=False, default=str)
            cell = ws.cell(row=row_idx, column=col_idx, value=value)
            cell.font = Font(name='Arial', size=9)
            cell.alignment = Alignment(vertical='top', wrap_text=False)
            if isinstance(value, float):
                cell.number_format = '#,##0.00'
            elif name.endswith('_PCT') or 'TAUX' in name:
                cell.number_format = '0.0000'
    ws.freeze_panes = 'A2'
    ws.auto_filter.ref = ws.dimensions
    ws.row_dimensions[1].height = 34
    for idx, name in enumerate(columns, start=1):
        if 'JSON' in name or 'RAW_TEXT' in name:
            width = 40
        elif 'ADRESSE' in name or 'RAISON_SOCIALE' in name or 'COMMENTAIRE' in name:
            width = 34
        elif 'DATE' in name or 'PERIODE' in name:
            width = 16
        elif 'MONTANT' in name or 'SALAIRE' in name or 'PART_' in name:
            width = 18
        else:
            width = min(28, max(12, len(name) + 2))
        ws.column_dimensions[get_column_letter(idx)].width = width
    return ws

def build_long_source_rows(dossiers):
    rows = []
    for dossier in dossiers:
        source_file = dossier['flat'].get('SRC_FICHIER')
        for page in dossier.get('page_records', []):
            normalized = page.get('normalized') or {}
            raw_data = page.get('raw_data') or {}
            keys = sorted(set(normalized) | set(raw_data))
            for key in keys:
                rows.append({'SRC_FICHIER': source_file, 'SRC_PAGE': page.get('page_num'), 'SRC_TYPE_DOCUMENT': page.get('doc_type'), 'SRC_CHAMP': key, 'SRC_VALEUR_BRUTE': raw_data.get(key), 'SRC_VALEUR_NORMALISEE': normalized.get(key), 'SRC_CONFIDENCE_CLASSIFICATION': page.get('classification_confidence'), 'SRC_CONFIDENCE_DOCUMENT': normalized.get('confidence_globale')})
    return rows

def create_excel(excel_path, dossiers, errors):
    dom_rows = [dict(d['flat']) for d in dossiers]
    dom_rows = add_renewal_links(dom_rows)
    by_source = {r.get('SRC_FICHIER'): r for r in dom_rows}
    for dossier in dossiers:
        source = dossier['flat'].get('SRC_FICHIER')
        if source in by_source:
            dossier['flat'] = by_source[source]
    planning_rows = []
    document_rows = []
    for dossier in dossiers:
        planning_rows.extend(dossier.get('planning_tl', []))
        source = dossier['flat'].get('SRC_FICHIER')
        for page in dossier.get('page_records', []):
            document_rows.append({'SRC_FICHIER': source, 'PAGE': page.get('page_num'), 'TYPE_DOCUMENT': page.get('doc_type'), 'CONFIDENCE_CLASSIFICATION': page.get('classification_confidence'), 'WHITE_RATIO': page.get('white_ratio'), 'WIDTH': page.get('width'), 'HEIGHT': page.get('height'), 'TOKENS_IN': page.get('classification_tokens_in', 0) + page.get('extraction_tokens_in', 0), 'TOKENS_OUT': page.get('classification_tokens_out', 0) + page.get('extraction_tokens_out', 0), 'EXTRACTION_JSON': json.dumps(page.get('normalized', {}), ensure_ascii=False, default=str), 'REPONSE_BRUTE': page.get('extraction_raw_text')})
    source_rows = build_long_source_rows(dossiers)
    params_rows = [{'PARAMETRE': 'MODEL_PATH', 'VALEUR': MODEL_PATH}, {'PARAMETRE': 'PDF_ZOOM', 'VALEUR': PDF_ZOOM}, {'PARAMETRE': 'IMAGE_MAX_SIZE', 'VALEUR': IMAGE_MAX_SIZE}, {'PARAMETRE': 'GPU_BATCH_SIZE_CLASSIFICATION', 'VALEUR': GPU_BATCH_SIZE_CLASSIFICATION}, {'PARAMETRE': 'GPU_BATCH_SIZE_EXTRACTION', 'VALEUR': GPU_BATCH_SIZE_EXTRACTION}, {'PARAMETRE': 'PRORATA_MODE', 'VALEUR': PRORATA_MODE}, {'PARAMETRE': 'VERSION_PIPELINE', 'VALEUR': PIPELINE_VERSION}]
    wb = Workbook()
    wb.remove(wb.active)
    sheet_from_rows(wb, 'DOMICILIATIONS', dom_rows)
    sheet_from_rows(wb, 'PLANNING_TL', planning_rows)
    sheet_from_rows(wb, 'DOCUMENTS', document_rows)
    sheet_from_rows(wb, 'CHAMPS_SOURCE', source_rows)
    sheet_from_rows(wb, 'ERREURS', errors)
    sheet_from_rows(wb, 'PARAMETRES', params_rows, ['PARAMETRE', 'VALEUR'])
    wb.save(excel_path)
    print(f'✅ Excel créé : {excel_path} | {len(dom_rows)} domiciliation(s) | {len(planning_rows)} période(s)')
print('✅ Export Excel OK')

In [ ]:

print(f"Nombre de PDF détectés : {len(pdfs)}")
if not pdfs:
    raise RuntimeError(
        f"Aucun PDF trouvé dans {INPUT_DIR}. "
        "Déposer les dossiers de domiciliation dans ce répertoire."
    )

diagnostic_errors = []
total_pages = 0

for p in pdfs:
    try:
        with fitz.open(str(p)) as doc:
            page_count = int(doc.page_count)
        total_pages += page_count
        print(
            f"{p.name} -> pages={page_count}, "
            f"taille={p.stat().st_size:,} octets"
        )
        if page_count <= 0:
            diagnostic_errors.append(f"{p.name}: 0 page")
    except Exception as exc:
        diagnostic_errors.append(f"{p.name}: {exc!r}")

if diagnostic_errors:
    raise RuntimeError(
        "Diagnostic PDF en erreur :\n- " + "\n- ".join(diagnostic_errors)
    )

print(f"✅ Diagnostic PDF : {len(pdfs)} fichier(s), {total_pages} page(s)")

# Nettoyage des anciens checkpoints vides uniquement.
removed = 0
for json_file in JSON_DIR.glob("*.json"):
    try:
        dossier = json.loads(json_file.read_text(encoding="utf-8"))
        pages_checkpoint = int(
            dossier.get("stats", {}).get("pages", 0) or 0
        )
        records_checkpoint = dossier.get("page_records") or []
        if pages_checkpoint <= 0 or not records_checkpoint:
            json_file.unlink()
            removed += 1
            print(f"Checkpoint vide supprimé : {json_file.name}")
    except Exception:
        # Un checkpoint illisible ne doit pas être repris.
        json_file.unlink()
        removed += 1
        print(f"Checkpoint illisible supprimé : {json_file.name}")

print(f"Checkpoints supprimés : {removed}")


In [ ]:

# Test technique sur le premier PDF avant le traitement complet.
_test_pdf = pdfs[0]
_test_pages = pdf_to_pages(_test_pdf)

print(
    f"Test conversion : {_test_pdf.name} -> "
    f"{len(_test_pages)} page(s)"
)
print(
    "Dimensions première page :",
    _test_pages[0]["width"],
    "x",
    _test_pages[0]["height"],
)
print(
    "Ratio blanc première page :",
    _test_pages[0]["white_ratio"],
)

if len(_test_pages) == 0:
    raise RuntimeError("Le test de conversion PDF a retourné zéro page.")

# Libération immédiate des images du test.
del _test_pages
gc.collect()
print("✅ Test de conversion PDF réussi")


## 13. Exécution complète avec reprise automatique

In [ ]:
ram_free = psutil.virtual_memory().available / 1000000000.0
log(f'RAM libre : {ram_free:.1f} GB')
existing_by_hash = {}
for json_file in JSON_DIR.glob('*.json'):
    try:
        with open(json_file, encoding='utf-8') as f:
            dossier = json.load(f)
        file_hash = dossier.get('flat', {}).get('SRC_SHA256')
        if file_hash:
            existing_by_hash[file_hash] = dossier
    except Exception as exc:
        log(f'⚠️ Checkpoint illisible {json_file.name}: {exc}')
all_dossiers = []
errors = []
for position, pdf_path in enumerate(pdfs, start=1):
    try:
        file_hash = sha256_file(pdf_path)
        if file_hash in existing_by_hash:
            dossier = existing_by_hash[file_hash]
            valid_checkpoint = bool(dossier.get('page_records')) and int(dossier.get('stats', {}).get('pages', 0) or 0) > 0
            if valid_checkpoint:
                all_dossiers.append(dossier)
                log(f'[{position}/{len(pdfs)}] ↩️ Reprise checkpoint valide : {pdf_path.name}')
                continue
            log(f'[{position}/{len(pdfs)}] ⚠️ Checkpoint vide ignoré : {pdf_path.name}')
        dossier = process_pdf(pdf_path)
        all_dossiers.append(dossier)
    except Exception as exc:
        error = {'SRC_FICHIER': pdf_path.name, 'ETAPE': 'PROCESS_PDF', 'ERREUR': repr(exc), 'DATE': datetime.now().isoformat(timespec='seconds')}
        errors.append(error)
        log(f'[{position}/{len(pdfs)}] ❌ {pdf_path.name}: {exc}')
    finally:
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
create_excel(EXCEL_PATH, all_dossiers, errors)
master_payload = {'generated_at': datetime.now().isoformat(timespec='seconds'), 'pipeline_version': PIPELINE_VERSION, 'prorata_mode_reference': PRORATA_MODE, 'domiciliations': [d.get('flat', {}) for d in all_dossiers], 'planning_tl': [row for d in all_dossiers for row in d.get('planning_tl', [])], 'errors': errors}
with open(MASTER_JSON_PATH, 'w', encoding='utf-8') as f:
    json.dump(master_payload, f, ensure_ascii=False, indent=2, default=str)
log(f'JSON maître : {MASTER_JSON_PATH}')
log(f'✅ Pipeline terminé | dossiers={len(all_dossiers)} | erreurs={len(errors)}')


## 14. Lecture des résultats

- `domiciliations_master.json` : référentiel consolidé avec les liens de renouvellement et tout le planning.
- `DOMICILIATIONS` : une ligne par contrat/domiciliation, avec informations KYC, père, mère, montants contractuels, permis, données brutes par document et montants P1/P2.
- `PLANNING_TL` : une ligne par période de transfert autorisée. Les mois partiels sont nommés, par exemple, `2025-06P2` et `2026-06P1`.
- `PLN_MONTANT_AUTORISE_SAISI` : colonne volontairement vide destinée à être complétée/validée avant les contrôles mensuels.
- `DOCUMENTS` : classification et extraction page par page.
- `CHAMPS_SOURCE` : traçabilité longue de toutes les valeurs brutes et normalisées.
- `ERREURS` : erreurs techniques.
- `PARAMETRES` : version du modèle et règles de génération.


In [ ]:
if EXCEL_PATH.exists():
    df_dom = pd.read_excel(EXCEL_PATH, sheet_name='DOMICILIATIONS')
    df_plan = pd.read_excel(EXCEL_PATH, sheet_name='PLANNING_TL')
    print(f'Domiciliations : {len(df_dom)}')
    print(f'Périodes TL    : {len(df_plan)}')
    display(df_dom.head(3))
    display(df_plan.head(12))
else:
    print("Le fichier Excel n'a pas encore été généré.")